In [1]:
import zipfile

with zipfile.ZipFile("/content/Assignment 2 files.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/")


# **Mapping each image to their corresponding captions**

---



this cell is for train.txt captions

In [1]:
from collections import defaultdict


train_captions = defaultdict(list) # will be used later

with open("/content/train.txt", 'r') as f:
    for line in f:
        img_name, caption = line.strip().split('\t')
        train_captions[img_name].append(caption)


for img, caps in list(train_captions.items())[:2]:  #this part of the code is just to show an example of the mapping for 2 pictures
    print(f"{img}:")
    for c in caps:
        print(f"  - {c}")

2874876837_80d178ba9b.jpg:
  - A boy bouncing on a trampoline .
  - A boy jumping on a trampoline .
  - A boy jumps on a trampoline .
  - A little boy in a white long sleeved t-shirt jumps on a trampoline .
  - Boy jumps on trampoline outside house .
636503038_17ca82b50f.jpg:
  - "Three fishers walk over rocks toward water , lake and cloudy sky in background ."
  - Three people are standing on rocks near a body of water .
  - Three people are walking on rocks towards a lake .
  - Three people stand with fishing poles on the rocks near the water .
  - "Three people walking along rocks , going to go fish"


this cell is for val.txt captions

In [2]:
val_captions = defaultdict(list) # will be used later

with open("/content/val.txt", 'r') as f:
    for line in f:
        img_name, caption = line.strip().split('\t')
        val_captions[img_name].append(caption)


for img, caps in list(val_captions.items())[:2]:
    print(f"{img}:")
    for c in caps:
        print(f"  - {c}")


3302804312_0272091cd5.jpg:
  - A black and brown dog is running with a tennis ball in its mouth .
  - A brown and white dog is running with a ball in its mouth on grass in front of a fence .
  - A brown and white dog runs beside a wooden fence with a tennis ball in his mouth .
  - A brown and white dog runs with a tennis ball in its mouth .
  - A brown dog is running by a wooden fence with a ball in its mouth .
607339469_af851c4119.jpg:
  - A girl in a purple shirt feeding ducks
  - A girl wearing purple lean over the edge to reach for the ducks .
  - "A little girl dressed in a pink shirt , blue shorts and a pink hair ribbon feeding ducks that are swimming in water ."
  - A little girl leans over the side of a wall to feed ducks in the water .
  - Little girl feeds the ducks .


# **Images preprocessing and feature extraction**

---



In [3]:
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import os
from PIL import Image

In [4]:
# Load InceptionV3 model
base_model = InceptionV3(weights='imagenet')
model = Model(inputs=base_model.input, outputs=base_model.get_layer('avg_pool').output) # here we cut off the last classification layer from InceptionV3 so that the model works like a cnn feature extractor

preprocess images function

In [5]:
def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(299, 299))  # Resize for InceptionV3 since it requires the size (299, 299)
    x = image.img_to_array(img)  # Convert image to numpy array
    x = np.expand_dims(x, axis=0)  # Add a batch dimension so that the shape changes from (299, 299, 3) to (1, 299, 299, 3), indicating that this batch has 1 image with size 299x299 and has 3 channels
    x = preprocess_input(x)  # preprocess_input normalizes the input for InceptionV3 by scaling RGB to [-1,1]
    return x


extract features function

In [6]:
def extract_features(folder_path, model):
    features = {}
    image_files = os.listdir(folder_path) # have image_files as an array that holds all the images that are inside a folder

    for i, img_name in enumerate(image_files):
        img_path = os.path.join(folder_path, img_name) # this makes it so that we can create the path of the specific image so we can access and preprocess it
        img_data = preprocess_image(img_path) # preprocess the image using the function we created. it returns the shape (1, 299, 299, 3)
        feature = model.predict(img_data) # get the feature (we modified the model so that it works as a feature extractor). it returns the shape (1, 2048)
        features[img_name] = feature #no flattening
        #features[img_name] = feature.reshape(-1) # flattens the data (remove the batch dimension so that the shape becomes (2048,) ) then record it as the feature for this image
        # we used .squeeze() instead of .flatten() to remove only the extra batch dimension turning it from (1, 2048) to (2048,) sincec .squeeze removes only the dimentions that equal 1
        # .flatten() was causing inconsistent shapes for some reason, leading to inhomogeneous shape error.
        # i will try .reshape since the problem occurred again
        #remove the flattening part. and if it didnt work try to use expand_dims on the inputs of the RNN model

    return features

In [ ]:
train_image_folder = "/content/train/train"
train_features = extract_features(train_image_folder, model)

Streaming output truncated to the last 5000 lines.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step

In [ ]:
val_image_folder = "/content/val/val"
val_features = extract_features(val_image_folder, model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 336ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 444ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 446ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 448ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [ ]:
import pickle

# save train features so that i dont have to wait a long time each time i run
with open("train_features.pkl", "wb") as f:
    pickle.dump(train_features, f)


In [7]:
import pickle
# load train features from the file
with open("train_features.pkl", "rb") as f:
    train_features = pickle.load(f)


In [ ]:
import pickle

# Save val features
with open("val_features.pkl", "wb") as f:
    pickle.dump(val_features, f)


In [8]:
# Load val features
with open("val_features.pkl", "rb") as f:
    val_features = pickle.load(f)


# **Caption Tokenization and Sequence Preparation**

---



In [9]:
all_captions = []

for img_captions in train_captions.values():
    for cap in img_captions:
        all_captions.append(f"<start> {cap.strip()} <end>") # .strip removes whitespaces from the caption
        # <start> helps the model know when to begin a sentence.
        # <end> helps the model know when to stop generating a sentence.


In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(oov_token="<unk>")
tokenizer.fit_on_texts(all_captions) #tokenize all the captions


In [11]:
vocab_size = len(tokenizer.word_index) + 1  # +1 for padding
print("Vocabulary size:", vocab_size)

Vocabulary size: 7779


In [12]:
train_seqs = tokenizer.texts_to_sequences(all_captions) # turn the captions to sequences (list of numbers)

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = max(len(seq) for seq in train_seqs) # get the length of the longest sequence for padding in the next cell
train_seqs = pad_sequences(train_seqs, maxlen=max_length, padding='post') # pad the other sequences so that they're as long as the longest sequence --> this will be used later, train_seqs was just an example of what will happen for all captions

print("Max sequence length:", max_length)


Max sequence length: 39


In [14]:
def create_sequences(tokenizer, max_length, captions_dict, image_features, vocab_size):
    X1, X2, y = [], [], [] # X1 is the image features. X2 is the caption so far. y is the next word
    for img_name, captions in captions_dict.items():
      feature = image_features.get(img_name) #get the feature for the image we will be working on in this loop --> X1

      if feature is None:
        # Skip if the image feature is not available
        continue
      # Clean the feature shape
      if feature.shape == (1, 2048):
          feature = np.squeeze(feature, axis=0) #if shape was (1, 2048) then turn it to (2048,) so that all features have the same shape and be able to be put together in a numpy array
      elif feature.shape == (2048,): #if not then do nothing
          pass
      else:
          raise ValueError(f"Unexpected feature shape: {feature.shape}")

      for caption in captions:
          seq = tokenizer.texts_to_sequences([f"<start> {caption.strip()} <end>"])[0] # turn the captions to sequences (list of numbers)

          for i in range(1, len(seq)):
              in_seq = seq[:i] # this is the partial caption so far (input) --> X2
              out_word = seq[i] # this is the next word to predict (output) --> y

              in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0] # here we pad using max_length from the last cell. pad all sequences so that they're as long as the longest sequence

              X1.append(feature) #the number of times the feature repeats depends on the sequence's length and the number of captions for this image
              X2.append(in_seq)
              y.append(out_word)

    return np.array(X1), np.array(X2), np.array(y).reshape(-1, 1)


In [15]:
X1_train, X2_train, y_train = create_sequences(
    tokenizer=tokenizer,
    max_length=max_length,
    captions_dict=train_captions,
    image_features=train_features,
    vocab_size=vocab_size
)

print("Image feature shape (X1):", X1_train.shape)
print("Input sequence shape (X2):", X2_train.shape)
print("Target word shape (y):", y_train.shape)


Image feature shape (X1): (382573, 2048)
Input sequence shape (X2): (382573, 39)
Target word shape (y): (382573, 1)


In [16]:
X1_val, X2_val, y_val = create_sequences(
    tokenizer=tokenizer,
    max_length=max_length,
    captions_dict=val_captions,
    image_features=val_features,
    vocab_size=vocab_size
)

print(f"Validation Data Shapes:")
print(f"Image feature shape (X1_val): {X1_val.shape}")
print(f"Input sequence shape (X2_val): {X2_val.shape}")
print(f"Target word shape (y_val): {y_val.shape}")


Validation Data Shapes:
Image feature shape (X1_val): (47570, 2048)
Input sequence shape (X2_val): (47570, 39)
Target word shape (y_val): (47570, 1)


# **Model Building**

---



In [17]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, Dropout, add

In [19]:
# Image feature input (X1)
image_input = Input(shape=(2048,), name="image_input")  # Input: extracted image feature (2048-dim)
img_dense = Dense(256, activation='relu')(image_input)  # Dense layer to transform image features

# Caption input (X2)
caption_input = Input(shape=(max_length,), name="caption_input")  # Input: padded sequence of words
caption_embedding = Embedding(input_dim=vocab_size, output_dim=256, mask_zero=True)(caption_input)  # Word embedding layer
caption_lstm = LSTM(256)(caption_embedding)  # LSTM layer to process the sequence

# Merge both image and caption branches
merged = add([img_dense, caption_lstm])  # Combine image and caption context
merged_dense = Dense(256, activation='relu')(merged)  # Further process merged features

# Output layer: predict the next word
outputs = Dense(vocab_size, activation='softmax')(merged_dense)  # Predict the next word from the vocabulary

# Define the final model
rnn_model = Model(inputs=[image_input, caption_input], outputs=outputs)

# Compile the model
rnn_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

# Print model summary
rnn_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ caption_input       │ (None, 39)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image_input         │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 39, 256)   │  1,991,424 │ caption_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 39)        │          0 │ caption_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256)       │    524,544 │ image_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 256)       │    525,312 │ embedding_1[0][0… │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 256)       │          0 │ dense_3[0][0],    │
│                     │                   │            │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │     65,792 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 7779)      │  1,999,203 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,106,275 (19.48 MB)

 Trainable params: 5,106,275 (19.48 MB)

 Non-trainable params: 0 (0.00 B)

train the model

In [21]:
from tensorflow.keras.callbacks import EarlyStopping
# Define early stopping, which stops training if the next coupel of epochs makes the loss worse
earlystop = EarlyStopping(
    monitor='val_loss',     # Watch validation loss
    patience=15,             # If val_loss doesn't improve for 15 epochs, stop
    restore_best_weights=True  # Restore model weights from best epoch
)

In [22]:
batch_size = 256
epochs = 100

# Train the model
history = rnn_model.fit(
    [X1_train, X2_train],  # Two inputs
    y_train,               # Target output
    batch_size=batch_size,
    epochs=epochs,
    validation_data=([X1_val, X2_val], y_val),
    callbacks=[earlystop],  # Add callback to early stop
    verbose=1  # Show progress
)


Epoch 1/100
1495/1495 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 4.4693 - val_loss: 3.3293
Epoch 2/100
1495/1495 ━━━━━━━━━━━━━━━━━━━━ 38s 13ms/step - loss: 3.0622 - val_loss: 3.1603
Epoch 3/100
1495/1495 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 2.7030 - val_loss: 3.1483
Epoch 4/100
1495/1495 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 2.4575 - val_loss: 3.1689
Epoch 5/100
1495/1495 ━━━━━━━━━━━━━━━━━━━━ 19s 13ms/step - loss: 2.2523 - val_loss: 3.2670
Epoch 6/100
1495/1495 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 2.0919 - val_loss: 3.3829
Epoch 7/100
1495/1495 ━━━━━━━━━━━━━━━━━━━━ 19s 13ms/step - loss: 1.9533 - val_loss: 3.4875
Epoch 8/100
1495/1495 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 1.8394 - val_loss: 3.6281
Epoch 9/100
1495/1495 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - loss: 1.7375 - val_loss: 3.7647
Epoch 10/100
1495/1495 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 1.6507 - val_loss: 3.8688
Epoch 11/100
1495/1495 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 1.5672 - val_loss: 4.01

In [23]:
#save the model in a file just in case
rnn_model.save("captioning_model.h5")

In [ ]:
#load the model
rnn_model = load_model("captioning_model.h5")

In [24]:
def generate_caption(model, tokenizer, photo_feature, max_length):
    in_text = '<start>'

    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length, padding='post')

        yhat = model.predict([photo_feature, sequence], verbose=0)
        yhat = np.argmax(yhat)
        word = tokenizer.index_word.get(yhat)

        if word is None:
            break

        if word == 'end':
            break  # Stop generating if the model predicts "end"

        in_text += ' ' + word

    # Remove <start> from generated caption
    final_caption = in_text.replace('<start>', '').strip()

    return final_caption


In [29]:
# Pick one image
image_name = list(train_features.keys())[0]
photo_feature = train_features[image_name]

# Make sure feature has batch dimension
if photo_feature.shape == (2048,):
    photo_feature = np.expand_dims(photo_feature, axis=0)

# Generate caption
caption = generate_caption(rnn_model, tokenizer, photo_feature, max_length)

print(f"Generated caption for image {image_name}:")
print(caption)


Generated caption for image 2998945968_86f236d1e8.jpg:
a group of people are riding a dirt bike


submission

In [31]:
import pandas as pd

# Extract features from test images
test_image_folder = "/content/test/test"
test_features = extract_features(test_image_folder, model)  # using your existing extract_features()

# Generate captions for each image using your generate_caption()
predictions = []

for img_name, feature in test_features.items():
    # Generate the caption for the image
    caption = generate_caption(rnn_model, tokenizer, feature, max_length)

    predictions.append((img_name, caption))

submission_df = pd.DataFrame(predictions, columns=["image_id", "caption"])
submission_df.to_csv("submission.csv", index=False)

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━